# 3_primary_translation_analysis
Inspect primary translation results before proceeding to evaluation phase

## List contents of */checkpoints* subdirectory

In [40]:
# show json files in the /checkpoints subdirectory
import importlib
import workflow_helpers
workflow_helpers = importlib.reload(workflow_helpers)

checkpoints_dir = workflow_helpers.get_checkpoints_dir()
json_files = workflow_helpers.list_json_files(checkpoints_dir)

Found JSON files:
- elements_batched_20260403_2247.json
- primary_retry_completed_1_20260403_2253.json
- primary_translation_completed_20260403_2251.json


## Load json state file
Enter the json filename you wish to analyze in the json_file variable in the cell below

In [41]:
# Load checkpoint state (metadata + elements)

import os
from importlib import reload
import workflow_helpers

workflow_helpers = reload(workflow_helpers)

# Define the checkpoint file
checkpoint_dir = "checkpoints"
json_file = "primary_retry_completed_1_20260403_2253.json"

# Full path
checkpoint_path = os.path.join(checkpoint_dir, json_file)

# Load metadata and elements
metadata, elements = workflow_helpers.load_elements_checkpoint(checkpoint_path)

print("Loaded checkpoint:", checkpoint_path)
print("Stage:", metadata.get("stage"))
print("Source language:", metadata.get("source_language"))
print("Target language:", metadata.get("target_language"))
print("Primary model:", metadata.get("primary_model_name"))
print(f"Loaded checkpoint with {len(elements)} elements.")
print("Example entry:")
elements[0]

Loaded checkpoint: checkpoints\primary_retry_completed_1_20260403_2253.json
Stage: primary_retry_completed
Source language: English
Target language: Traditional Chinese
Primary model: gemini-3.1-pro-preview
Loaded checkpoint with 29 elements.
Example entry:


{'element_number': 1,
 'element_type': 'paragraph',
 'word_style': 'h1',
 'text': 'Introduction',
 'element_id': '8633e724fc99',
 'tokens': 1,
 'batch_number': 1,
 'primary_translation': '簡介',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

## Analysis
### Query for unchanged elements

In [42]:
unchanged = [
    el for el in elements
    if isinstance(el.get("text"), str)
    and isinstance(el.get("primary_translation"), str)
    and el["text"].strip() == el["primary_translation"].strip()
]

len(unchanged)

0

In [43]:
for el in unchanged[:10]:
    print(f"ID: {el['element_id']}")
    print(f"EN: {el['text']}")
    print(f"RU: {el['primary_translation']}")
    print("-" * 60)

In [44]:
# create df for the unchanged entries
import pandas as pd

df_unchanged = pd.DataFrame(unchanged)
df_unchanged.head(10)


""


### Inspect 'primary_translation' for likely English
#### heuristic method (ascii detection)

In [45]:
import pandas as pd

def is_likely_english(text, min_alpha=10, ascii_ratio_threshold=0.85):
    """
    Heuristic, target-language-agnostic:
    - Ignore very short strings with < min_alpha alphabetic chars.
    - Compute fraction of alphabetic chars that are ASCII a–z.
    - If that fraction >= ascii_ratio_threshold, flag as likely English.
    """
    if not isinstance(text, str):
        return False

    letters = [ch for ch in text if ch.isalpha()]
    if len(letters) < min_alpha:
        return False  # too short / code-like; we don't care

    ascii_letters = sum("a" <= ch.lower() <= "z" for ch in letters)
    ratio = ascii_letters / len(letters)

    return ratio >= ascii_ratio_threshold

# Build list of elements where primary_translation is probably English
suspect_english = [
    el for el in elements
    if is_likely_english(el.get("primary_translation", ""))
]

len(suspect_english)

0

In [46]:
## uncomment if suspect_english is >0
# for el in suspect_english[:20]:
#     print(f"ID:  {el['element_id']}")
#     print(f"EN:  {el['text']}")
#     print(f"TR:  {el['primary_translation']}")
#     print("-" * 80)

In [47]:
## uncomment to create a df of suspect_english
# df_suspect_english = pd.DataFrame(suspect_english)
# df_suspect_english.head(20)

#### Langid method

In [48]:
import langid
import pandas as pd

# Optional: let langid use full language set (future-proof)
# langid.set_languages(None)

suspect_langid_en = []

for el in elements:
    txt = el.get("primary_translation", "")
    if not isinstance(txt, str) or not txt.strip():
        continue

    letters = [ch for ch in txt if ch.isalpha()]
    if len(letters) < 10:
        continue  # skip ultra-short things

    lang, score = langid.classify(txt)
    if lang == "en":
        el_copy = el.copy()
        el_copy["detected_lang"] = lang
        el_copy["lang_score"] = score
        suspect_langid_en.append(el_copy)

len(suspect_langid_en)

0

In [49]:
## uncomment to create df of suspect_langid
# df_suspect_langid_en = pd.DataFrame(suspect_langid_en)
# df_suspect_langid_en.head(20)

In [50]:
## uncomment if more than 0 results 
## intersect both methods

# ids_heuristic = {el["element_id"] for el in suspect_english}
# high_conf_suspect = [
#     el for el in suspect_langid_en if el["element_id"] in ids_heuristic
# ]

# len(high_conf_suspect)

### Formatting and style analytics

In [51]:
# convert elements to df
import pandas as pd
df = pd.DataFrame(elements)

In [52]:
# Any missing or blank primary_translation?
mask_empty = df['primary_translation'].isna() | (df['primary_translation'].str.strip() == "")
df_empty = df[mask_empty]
len(df_empty), df_empty.head(10)

(0,
 Empty DataFrame
 Columns: [element_number, element_type, word_style, text, element_id, tokens, batch_number, primary_translation, primary_translation_model, primary_error, evaluator_ran, evaluator_passed, evaluator_feedback, evaluator_error, fallback_translation, fallback_translation_model, fallback_error, final, final_model]
 Index: [])

In [53]:
# Any unusual trunctions or expansions?
df['len_src'] = df['text'].str.len()
df['len_tr'] = df['primary_translation'].str.len()

# Avoid division by zero
df['len_ratio'] = df['len_tr'] / df['len_src'].replace({0: pd.NA})

# Suspiciously short translations for non-trivial source
suspect_short = df[(df['len_src'] >= 40) & (df['len_ratio'] < 0.4)]

# Suspiciously long translations (might be okay, but worth a look)
suspect_long  = df[(df['len_src'] >= 40) & (df['len_ratio'] > 3.0)]
len(suspect_short), len(suspect_long)

(20, 0)

In [54]:
# uncomment if results are greater than 0
suspect_short[['element_number', 'text', 'primary_translation']].head(20)
suspect_long[['element_number', 'text', 'primary_translation']].head(20)

,element_number,text,primary_translation


In [55]:
# inspect new line characters
df['nl_src'] = df['text'].str.count('\n')
df['nl_tr']  = df['primary_translation'].str.count('\n')

suspect_newlines = df[(df['nl_src'] != df['nl_tr']) & (df['nl_src'] > 0)]
len(suspect_newlines)

0

In [56]:
# suspect_newlines[['element_number', 'text', 'primary_translation']].head(20)

In [57]:
# Inspect * marker counts
import re
df['stars_src'] = df['text'].str.count(r'\*')
df['stars_tr']  = df['primary_translation'].str.count(r'\*')

star_mismatch = df[df['stars_src'] != df['stars_tr']]
len(star_mismatch)

0

In [58]:
star_mismatch[['element_number', 'text', 'primary_translation']].head(20)

,element_number,text,primary_translation


In [59]:
# select an index number to inspect
elements[10]

{'element_number': 11,
 'element_type': 'paragraph',
 'word_style': 'quotation',
 'text': 'You will need a **Bible** and a **pencil** or **highlighter**.',
 'element_id': 'fb49158b9536',
 'tokens': 19,
 'batch_number': 4,
 'primary_translation': '你需要一本**聖經**，以及一枝**鉛筆**或**螢光筆**。',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

In [60]:
# inspect URLs
df['has_url_src'] = df['text'].str.contains(r'http[s]?://', regex=True, na=False)
df['has_url_tr']  = df['primary_translation'].str.contains(r'http[s]?://', regex=True, na=False)

url_lost = df[(df['has_url_src']) & (~df['has_url_tr'])]
len(url_lost)

0

In [61]:
# url_lost[['element_number', 'text', 'primary_translation']].head(20)

In [62]:
# ordered list inspection (may miss some ol items)

# Very simple: lines starting with digit + '.'
df['starts_with_num_src'] = df['text'].str.match(r'^\s*\d+\.', na=False)
df['starts_with_num_tr']  = df['primary_translation'].str.match(r'^\s*\d+\.', na=False)

num_list_mismatch = df[df['starts_with_num_src'] & ~df['starts_with_num_tr']]
len(num_list_mismatch)

0

In [63]:
## uncomment to show
# num_list_mismatch[['element_number', 'text', 'primary_translation']].head(20)

In [64]:
# summarize results
summary = {
    "total_elements": len(df),
    "missing_or_blank_translation": int(mask_empty.sum()),
    "with_primary_error": int(df['primary_error'].notna().sum()),
    "suspect_short_len": int(len(suspect_short)),
    "suspect_long_len": int(len(suspect_long)),
    "newline_mismatch": int(len(suspect_newlines)),
    "star_mismatch": int(len(star_mismatch)),
    "url_lost": int(len(url_lost)),
}
summary

{'total_elements': 29,
 'missing_or_blank_translation': 0,
 'with_primary_error': 0,
 'suspect_short_len': 20,
 'suspect_long_len': 0,
 'newline_mismatch': 0,
 'star_mismatch': 0,
 'url_lost': 0}

## Langid classification

In [65]:
# langid detect language random sample (important to realize that langid is leaky)

import langid

# Sample a subset to keep it quick
sample = df.sample(min(300, len(df)), random_state=42).copy()

def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"
    return langid.classify(text)[0]

sample['detected_lang'] = sample['primary_translation'].apply(detect_lang_safe)
sample['detected_lang'].value_counts()

detected_lang
zh    29
Name: count, dtype: int64

In [66]:
# inspect all elements

import langid
import pandas as pd

df = pd.DataFrame(elements)

def detect_lang_safe(text):
    if not isinstance(text, str) or not text.strip():
        return "unknown"
    return langid.classify(text)[0]

# Run on the full set
df['detected_lang'] = df['primary_translation'].apply(detect_lang_safe)

df['detected_lang'].value_counts()

detected_lang
zh    29
Name: count, dtype: int64

In [67]:
df_problem = df[df['detected_lang'].isin(['en','zh','fr','ka','he','lv'])]
df_problem[['element_number','word_style','text','primary_translation']]

,element_number,word_style,text,primary_translation
0,1,h1,Introduction,簡介
1,2,h2,Welcome,歡迎
2,3,body,This course is designed for the person that wa...,本課程專為那些想了解成為或作為耶穌跟隨者有何意義的人而設計。許多尋求這方面知識的人明白他們必...
3,4,body,This is a **self-directed study** to assist yo...,這是一份**自學指南**，旨在協助您尋求答案。它的目的是帶您快速瀏覽精選的聖經書卷，為您提供...
4,5,body,The course consists of **two parts**: the firs...,這門課程包含**兩個部分**：第一部分涵蓋基本內容，可以在兩週內完成。第二部分「深入探究」聖...
5,6,body,"After completing this course, you will be **eq...",完成這門課程後，你將會**得到裝備、充滿力量並滿懷熱情**地繼續研讀聖經。
6,7,h2,The Non-Sequential Reading Approach,非順序閱讀法
7,8,body,Most people that are new to the Bible approach...,大多數初接觸聖經的人會像閱讀其他書一樣來讀它——他們從頭開始，然後按順序一直讀到最後。
8,9,body,"However, this often leads to frustration becau...",然而，這通常會導致挫折，因為直線式閱讀法並不能輕易地幫助你在***閱讀的當下***理解聖經。...
9,10,h2,Getting Started,準備開始


In [68]:
# Check elements flagged as English

import langid

suspected_english = []

for el in elements:
    tr = el['primary_translation']
    lang, conf = langid.classify(tr)
    if lang == 'en' and len(tr) > 20:  # avoid tiny text
        suspected_english.append((el['element_number'], tr, conf))

len(suspected_english)

0

In [69]:
suspected_english

[]

In [70]:
# inspect an element by index number
elements[9]

{'element_number': 10,
 'element_type': 'paragraph',
 'word_style': 'h2',
 'text': 'Getting Started',
 'element_id': 'f1e071f3c3e5',
 'tokens': 2,
 'batch_number': 4,
 'primary_translation': '準備開始',
 'primary_translation_model': 'gemini-3.1-pro-preview',
 'primary_error': None,
 'evaluator_ran': None,
 'evaluator_passed': None,
 'evaluator_feedback': None,
 'evaluator_error': None,
 'fallback_translation': None,
 'fallback_translation_model': None,
 'fallback_error': None,
 'final': None,
 'final_model': None}

# Create a Word document based on primary translation only
- Save back to Word using primary translation only (no evaluation and finalization stages)
- Uses a standalone definition function rather than calling the function from workflow_helpers.py
- Saves a new Word document with the original English first and the translated language appended to the end of the document (non-interlinear)

### Function definitions

In [71]:
from docx import Document
from docx.shared import Inches
import re
import datetime
from pathlib import Path

# --- tiny inline markdown → runs (bold/italic only) ---
def markdown_to_runs(paragraph, text: str):
    # split on **bold** or *italic*
    tokens = re.split(r'(\*\*.*?\*\*|\*.*?\*)', text)
    for tok in tokens:
        if tok.startswith("**") and tok.endswith("**"):
            run = paragraph.add_run(tok[2:-2])
            run.bold = True
        elif tok.startswith("*") and tok.endswith("*"):
            run = paragraph.add_run(tok[1:-1])
            run.italic = True
        else:
            paragraph.add_run(tok)

# --- pick first available style from a list of candidates ---
def pick_style(doc: Document, *names: str, default="Normal") -> str:
    style_names = {s.name for s in doc.styles}
    for n in names:
        if n and n in style_names:
            return n
    return default

# --- classify a single logical line: blockquote / ul / ol / plain ---
_line_rx_ul = re.compile(r'^\s*[-*]\s+')
_line_rx_ol = re.compile(r'^\s*\d+\.\s+')
_line_rx_bq = re.compile(r'^\s*>\s+')

def classify_line(text: str):
    if _line_rx_bq.match(text):
        clean = _line_rx_bq.sub("", text, count=1)
        return ("blockquote", clean)
    if _line_rx_ul.match(text):
        clean = _line_rx_ul.sub("", text, count=1)
        return ("ul", clean)
    if _line_rx_ol.match(text):
        clean = _line_rx_ol.sub("", text, count=1)
        return ("ol", clean)
    return ("plain", text)

def export_elements_to_docx(
    elements,
    out_path="final_translation.docx",
    template_path=None,
    *,
    # style candidates to try (first found wins)
    ul_styles=("List Bullet", "ListParagraph", "List Paragraph"),
    ol_styles=("w2w_EN.NormListNumbered", "List Number", "List Numbered", "w2w_EN.NormList"),
    bq_styles=("Quote", "Intense Quote", "Block Text"),
    default_style_fallback="Normal",
    text_field="primary_translation",
):
    """
    Rebuild a Word doc from `elements`:
      - uses `primary_translation` by default (falls back to `text`)
      - paragraph style from `word_style` (if present in template)
      - special handling for blockquote/ul/ol markers in text
      - inline **bold** / *italic* preserved
    """
    doc = Document(template_path) if template_path else Document()

    # cache style picks
    ul_style = pick_style(doc, *ul_styles, default=default_style_fallback)
    ol_style = pick_style(doc, *ol_styles, default=default_style_fallback)
    bq_style = pick_style(doc, *bq_styles, default=default_style_fallback)

    for e in elements:
        # Prefer translated text, fall back to original
        raw = e.get(text_field) or e.get("text") or ""
        if not raw.strip():
            continue

        desired_style = e.get("word_style") or default_style_fallback
        style_exists = desired_style in {s.name for s in doc.styles}
        para_style_default = desired_style if style_exists else default_style_fallback

        # In most pipelines each element is one logical paragraph,
        # but we still split on hard newlines just in case.
        for line in raw.splitlines() or [""]:
            kind, clean = classify_line(line)

            if kind == "blockquote":
                p = doc.add_paragraph(style=bq_style)
                # If template lacks a quote style, indent a bit
                if bq_style == default_style_fallback:
                    p.paragraph_format.left_indent = Inches(0.25)
                markdown_to_runs(p, clean)
                continue

            if kind == "ul":
                p = doc.add_paragraph(style=ul_style)
                # If no bullet style exists, prepend a bullet char as last resort
                if ul_style == default_style_fallback:
                    markdown_to_runs(p, "• ")
                markdown_to_runs(p, clean)
                continue

            if kind == "ol":
                p = doc.add_paragraph(style=ol_style)
                # If no numbering style, prefix with a plain "1. " as last resort
                if ol_style == default_style_fallback:
                    markdown_to_runs(p, "1. ")
                markdown_to_runs(p, clean)
                continue

            # plain paragraph: use the original Word style if present
            p = doc.add_paragraph(style=para_style_default)
            markdown_to_runs(p, clean)

    doc.save(out_path)
    print(f"💾 Saved reconstructed Word file → {out_path}")

# ---------- Output path helper using fixed base name ----------

def build_output_path_from_base(docxfilename: str, translated_language: str, text_field: str) -> Path:
    """
    Build output filename from the source DOCX filename,
    plus translated language, text field label, and timestamp.

    Example:
      docxfilename='Galatians_Third_Edition_A4.docx', translated_language='Russian'
      -> Galatians_Third_Edition_A4_Russian_primary_translation_20251120_183155.docx
    """
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    base = Path(docxfilename).stem
    return Path(f"{base}_{translated_language}_{text_field}_{ts}.docx")

### Read original word filename and target language from metadata header

In [72]:
docxfilename = metadata.get("docxfilename")
target_language = metadata.get("target_language")

missing = [
    name for name, value in {
        "docxfilename": docxfilename,
        "target_language": target_language,
    }.items()
    if not value
]

if missing:
    raise ValueError(
        f"Missing required metadata field(s) in checkpoint: {', '.join(missing)}"
    )

print("Word source filename:", docxfilename)
print("Target language:", target_language)

Word source filename: custom_word_styles_example.docx
Target language: Traditional Chinese


### Call the function
- **Important:** Double-check the filename and translated language parameters below
- Uses the 'primary_translation' field by default (since the process has not progressed to the evaluation and final stages yet)

In [73]:
from pathlib import Path

# ---------------------------------------------
# Directory containing the Word files
# ---------------------------------------------
word_dir = Path("word_files")

# ---------------------------------------------
# Metadata-driven settings
# ---------------------------------------------
translated_language = target_language
text_field = "primary_translation"

template_path = word_dir / docxfilename

if not template_path.exists():
    raise FileNotFoundError(f"Template file not found: {template_path}")

# Output file will also go into word_files/
out_path = word_dir / build_output_path_from_base(
    docxfilename=docxfilename,
    translated_language=translated_language,
    text_field=text_field,
)

print("Template DOCX:", template_path)
print("Saving output to:", out_path)

# ---------------------------------------------
# Generate translated DOCX
# ---------------------------------------------
export_elements_to_docx(
    elements,
    out_path=out_path,
    template_path=template_path,
    text_field=text_field,
)

print("Done.")

Template DOCX: word_files\custom_word_styles_example.docx
Saving output to: word_files\custom_word_styles_example_Traditional Chinese_primary_translation_20260403_225429.docx
💾 Saved reconstructed Word file → word_files\custom_word_styles_example_Traditional Chinese_primary_translation_20260403_225429.docx
Done.
